# P3-v2 owner-frozen backbone execution

Run `checkpoint_smoke` once per backbone, inspect both smoke receipts, then switch to `full_units` once per backbone. The frozen source selection and six construct-cell counts are checked before any forecast call. Outputs are private in Google Drive; this notebook never computes an aggregate paper decision. A disconnect can be resumed by rerunning all cells with the same settings. TimesFM-3 has a non-commercial license; use it only within that license.


In [ ]:
BACKBONE = 'chronos_2'  # then 'timesfm_3'
MODE = 'checkpoint_smoke'  # after this backbone's smoke succeeds: 'full_units'
assert BACKBONE in {'chronos_2', 'timesfm_3'}
assert MODE in {'checkpoint_smoke', 'full_units'}
print('P3-v2:', BACKBONE, MODE)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, shutil, subprocess, sys

REPO = Path('/content/tsfm-covariate-faithfulness')
REPO_URL = 'https://github.com/FlyMe2star/tsfm-covariate-faithfulness.git'
if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'main'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', 'main', REPO_URL, str(REPO)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'requirements/p3-preflight.txt')], cwd=REPO, check=True)
model_req = REPO / ('requirements/chronos2.txt' if BACKBONE == 'chronos_2' else 'requirements/timesfm3.txt')
if BACKBONE == 'timesfm_3':
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'timesfm'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(model_req)], check=True)
src_path = str(REPO / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
import torch
assert torch.cuda.is_available(), 'Select a T4 or A100 GPU and restart this runtime.'
print('Git commit:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip())
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
from huggingface_hub import hf_hub_download
from covfaith_p3.data import sha256_file
from covfaith_p3_v2_model.runner import load_frozen_contract

config, construct_receipt = load_frozen_contract(REPO)
DATA_ROOT = Path('/content/drive/MyDrive/tsfm-covariate-faithfulness/p3_v2_source_data')
OUTPUT_ROOT = Path('/content/drive/MyDrive/tsfm-covariate-faithfulness/p3_v2_model_v1')
DATA_ROOT.mkdir(parents=True, exist_ok=True)
for source in config['data']['sources']:
    destination = DATA_ROOT / f"{source['config']}.parquet"
    if not destination.is_file() or sha256_file(destination) != source['parquet_sha256']:
        downloaded = hf_hub_download(
            repo_id=config['data']['repository'],
            repo_type=config['data']['repository_type'],
            revision=config['data']['revision'],
            filename=source['parquet_path'],
        )
        assert sha256_file(Path(downloaded)) == source['parquet_sha256'], f"{source['id']}: download hash mismatch"
        shutil.copyfile(downloaded, destination)
    assert sha256_file(destination) == source['parquet_sha256']
    print('Pinned source ready:', source['id'])
print('Approved construct gate:', construct_receipt['decision']['construct_gate_passed'])


In [ ]:
from covfaith.adapters import Chronos2Adapter, TimesFM3Adapter
from covfaith_p3_v2_model.runner import run_backbone_units
import time

model = next(item for item in config['models'] if item['id'] == BACKBONE)
load_start = time.perf_counter()
if BACKBONE == 'chronos_2':
    adapter = Chronos2Adapter.from_pretrained(model['checkpoint'], model['revision'], device='cuda', batch_size=128)
else:
    adapter = TimesFM3Adapter.from_pretrained(model['checkpoint'], model['revision'], device='cuda', per_core_batch_size=16)
model_load_seconds = time.perf_counter() - load_start
report = run_backbone_units(REPO, adapter, DATA_ROOT, OUTPUT_ROOT, mode=MODE, model_load_seconds=model_load_seconds)
assert report['scientific_gate_computed'] is False
expected_units = 1 if MODE == 'checkpoint_smoke' else 18
assert report['completed_unit_count'] == expected_units
assert report['valid_scenario_count'] == (1 if MODE == 'checkpoint_smoke' else 571)
print(json.dumps(report, indent=2, ensure_ascii=False))
print('Private artifacts:', OUTPUT_ROOT / MODE / BACKBONE)
